# Global Internet & Social Media Usage 2000-2023
### 200+ Countries | 24 Years | EDA + Machine Learning
**Author:** Hassan Ali | [Kaggle: hassanali789](https://www.kaggle.com/hassanali789)

This notebook analyzes 24 years of global internet usage data covering 200+ countries from 2000 to 2023. We explore the digital divide, regional trends, fastest growing nations, and build a model to predict internet penetration levels.

**Sources:** World Bank (IT.NET.USER.ZS, SP.POP.TOTL) · DataReportal 2023

---


## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette("husl")

print("All libraries loaded successfully!")

## 2. Load and Inspect Dataset

In [ ]:
df = pd.read_csv("/kaggle/input/global-internet-social-media-usage-2000-2023/global_internet_social_media_2000_2023.csv")

print(f"Shape       : {df.shape}")
print(f"Countries   : {df['country'].nunique()}")
print(f"Year range  : {df['year'].min()} to {df['year'].max()}")
print(f"Regions     : {df['region'].unique()}")
print(f"\nMissing values:")
print(df.isnull().sum())
df.head(10)

## 3. Statistical Summary

In [ ]:
print("=== Overall Statistics ===")
print(df[["internet_users_pct","internet_users_count","population"]].describe().round(2))

print("\n=== Records per Region ===")
print(df.groupby("region")["country"].nunique().sort_values(ascending=False))

print("\n=== Global Internet Penetration 2023 ===")
latest = df[df["year"] == 2023].dropna(subset=["internet_users_pct"])
print(f"Global average : {latest['internet_users_pct'].mean():.1f}%")
print(f"Highest        : {latest.nlargest(1,'internet_users_pct')[['country','internet_users_pct']].values[0]}")
print(f"Lowest         : {latest.nsmallest(1,'internet_users_pct')[['country','internet_users_pct']].values[0]}")

## 4. Global Internet Growth Over Time

In [ ]:
global_avg = df.groupby("year")["internet_users_pct"].mean().reset_index()

fig, ax = plt.subplots()
ax.fill_between(global_avg["year"], global_avg["internet_users_pct"],
                alpha=0.2, color="#1565C0")
ax.plot(global_avg["year"], global_avg["internet_users_pct"],
        color="#1565C0", linewidth=2.5, marker="o", markersize=4)
ax.set_title("Global Average Internet Penetration (2000-2023)", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Internet Users (%)")
ax.set_xlabel("Year")
ax.set_xticks(range(2000, 2024, 2))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))
plt.tight_layout()
plt.savefig("global_growth.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"2000 global avg : {global_avg[global_avg['year']==2000]['internet_users_pct'].values[0]:.1f}%")
print(f"2023 global avg : {global_avg[global_avg['year']==2023]['internet_users_pct'].values[0]:.1f}%")

## 5. Internet Penetration by Region

In [ ]:
region_year = df.groupby(["region","year"])["internet_users_pct"].mean().reset_index()

fig, ax = plt.subplots()
for region in sorted(df["region"].dropna().unique()):
    d = region_year[region_year["region"] == region]
    ax.plot(d["year"], d["internet_users_pct"], linewidth=2, marker="o",
            markersize=3, label=region)

ax.set_title("Internet Penetration by Region (2000-2023)", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Internet Users (%)")
ax.set_xlabel("Year")
ax.set_xticks(range(2000, 2024, 2))
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))
plt.tight_layout()
plt.savefig("region_trends.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Top 20 Countries by Internet Penetration (2023)

In [ ]:
latest = df[df["year"] == 2023].dropna(subset=["internet_users_pct"])
top20 = latest.nlargest(20, "internet_users_pct")
bot20 = latest.nsmallest(20, "internet_users_pct")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

bars = axes[0].barh(top20["country"], top20["internet_users_pct"],
                    color="#1565C0", edgecolor="white", linewidth=0.3)
axes[0].bar_label(bars, fmt="%.1f%%", padding=3, fontsize=8)
axes[0].set_title("Top 20 Countries — Internet Penetration 2023", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Internet Users (%)")
axes[0].invert_yaxis()

bars = axes[1].barh(bot20["country"], bot20["internet_users_pct"],
                    color="#E53935", edgecolor="white", linewidth=0.3)
axes[1].bar_label(bars, fmt="%.1f%%", padding=3, fontsize=8)
axes[1].set_title("Bottom 20 Countries — Internet Penetration 2023", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Internet Users (%)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig("top_bottom_countries.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. The Digital Divide — Regional Comparison 2023

In [ ]:
region_2023 = df[df["year"]==2023].dropna(subset=["internet_users_pct"])
region_avg = region_2023.groupby("region")["internet_users_pct"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(region_avg)))[::-1]
bars = ax.bar(region_avg.index, region_avg.values,
              color=colors, edgecolor="white", linewidth=0.3)
ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
ax.axhline(region_avg.mean(), color="black", linestyle="--",
           linewidth=1, label=f"Global avg: {region_avg.mean():.1f}%")
ax.set_title("Average Internet Penetration by Region (2023)", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Internet Users (%)")
ax.tick_params(axis="x", rotation=20)
ax.legend()
plt.tight_layout()
plt.savefig("digital_divide.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Fastest Growing Countries (2000-2023)

In [ ]:
growth = []
for country in df["country"].unique():
    c_df = df[df["country"] == country]
    y2000 = c_df[c_df["year"]==2000]["internet_users_pct"].values
    y2023 = c_df[c_df["year"]==2023]["internet_users_pct"].values
    if len(y2000) and len(y2023) and not pd.isna(y2000[0]) and not pd.isna(y2023[0]):
        growth.append({
            "country": country,
            "pct_2000": y2000[0],
            "pct_2023": y2023[0],
            "growth": y2023[0] - y2000[0]
        })

growth_df = pd.DataFrame(growth).sort_values("growth", ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
top15 = growth_df.head(15)
bars = ax.barh(top15["country"], top15["growth"],
               color="#43A047", edgecolor="white", linewidth=0.3)
ax.bar_label(bars, fmt="+%.1f%%", padding=3, fontsize=8)
ax.set_title("Top 15 Fastest Growing Internet Nations (2000-2023)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Percentage Point Growth")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("fastest_growing.png", dpi=150, bbox_inches="tight")
plt.show()
print(growth_df.head(10).to_string(index=False))

## 9. Social Media Penetration vs Internet Penetration (2023)

In [ ]:
sm_df = df[df["year"]==2023].dropna(subset=["social_media_pct_2023","internet_users_pct"])

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(sm_df["internet_users_pct"], sm_df["social_media_pct_2023"],
                     alpha=0.7, s=60, c=sm_df["internet_users_pct"], cmap="YlOrRd")
plt.colorbar(scatter, ax=ax, label="Internet Penetration %")

for _, row in sm_df.iterrows():
    ax.annotate(row["country"], (row["internet_users_pct"], row["social_media_pct_2023"]),
                fontsize=7, alpha=0.7, xytext=(3, 3), textcoords="offset points")

ax.set_title("Social Media vs Internet Penetration (2023)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Internet Users (%)")
ax.set_ylabel("Social Media Penetration (%)")

corr = sm_df[["internet_users_pct","social_media_pct_2023"]].corr().iloc[0,1]
ax.text(0.05, 0.95, f"Correlation: {corr:.3f}", transform=ax.transAxes,
        fontsize=10, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plt.savefig("social_vs_internet.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Heatmap — Internet Penetration by Region Over Time

In [ ]:
pivot = df.groupby(["region","year"])["internet_users_pct"].mean().reset_index()
pivot = pivot.pivot(index="region", columns="year", values="internet_users_pct")
pivot = pivot[[c for c in pivot.columns if c % 2 == 0]]  # Even years only

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=0.3, linecolor="white",
            cbar_kws={"label": "Internet Users (%)"}, ax=ax)
ax.set_title("Internet Penetration by Region and Year (%)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Year")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("region_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Pakistan vs World — Internet Growth

In [ ]:
pak = df[df["country"]=="Pakistan"].sort_values("year")
global_avg = df.groupby("year")["internet_users_pct"].mean().reset_index()

fig, ax = plt.subplots()
ax.plot(global_avg["year"], global_avg["internet_users_pct"],
        color="#1565C0", linewidth=2, label="Global Average", linestyle="--")
ax.plot(pak["year"], pak["internet_users_pct"],
        color="#43A047", linewidth=2.5, marker="o", markersize=4, label="Pakistan")

ax.set_title("Pakistan Internet Penetration vs Global Average", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Internet Users (%)")
ax.set_xlabel("Year")
ax.legend()
ax.set_xticks(range(2000, 2024, 2))
plt.tight_layout()
plt.savefig("pakistan_vs_world.png", dpi=150, bbox_inches="tight")
plt.show()

print("Pakistan internet stats:")
print(pak[["year","internet_users_pct","internet_users_count"]].tail(10).to_string(index=False))

## 12. Feature Engineering for Machine Learning

In [ ]:
df_ml = df.dropna(subset=["internet_users_pct","population"]).copy()
df_ml = df_ml.sort_values(["country","year"])

le = LabelEncoder()
df_ml["region_enc"] = le.fit_transform(df_ml["region"].fillna("Other"))

df_ml["lag1_pct"]   = df_ml.groupby("country")["internet_users_pct"].shift(1)
df_ml["lag2_pct"]   = df_ml.groupby("country")["internet_users_pct"].shift(2)
df_ml["growth_1yr"] = df_ml["internet_users_pct"] - df_ml["lag1_pct"]
df_ml["log_pop"]    = np.log1p(df_ml["population"])
df_ml["year_norm"]  = df_ml["year"] - 2000

df_ml = df_ml.dropna(subset=["lag1_pct","lag2_pct","growth_1yr"])

print(f"ML-ready rows : {len(df_ml):,}")
df_ml[["country","year","internet_users_pct","lag1_pct","growth_1yr"]].head(10)

## 13. Model Training — Predicting Internet Penetration

In [ ]:
features = ["lag1_pct","lag2_pct","growth_1yr","log_pop","year_norm","region_enc"]
target   = "internet_users_pct"

X = df_ml[features].fillna(0)
y = df_ml[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Train size: {X_train.shape[0]:,}")
print(f"Test size : {X_test.shape[0]:,}")
print()

models = {
    "Linear Regression"  : LinearRegression(),
    "Random Forest"      : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting"  : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = []
trained = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2   = r2_score(y_test, preds)
    results.append({"Model": name, "MAE": round(mae,3), "RMSE": round(rmse,3), "R2": round(r2,4)})
    trained[name] = (model, preds)
    print(f"{name:22s} -> MAE: {mae:.2f}% | RMSE: {rmse:.2f}% | R2: {r2:.4f}")

## 14. Model Comparison and Feature Importance

In [ ]:
results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
best_name  = results_df.iloc[0]["Model"]
best_preds = trained[best_name][1]
best_model = trained[best_name][0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(results_df))
axes[0].bar(results_df["Model"], results_df["R2"],
            color=["#43A047","#1E88E5","#FB8C00"],
            edgecolor="white", linewidth=0.3)
for i, (_, row) in enumerate(results_df.iterrows()):
    axes[0].text(i, row["R2"]+0.005, f"R2={row['R2']}", ha="center", fontsize=9)
axes[0].set_title("Model R2 Score Comparison", fontsize=13, fontweight="bold")
axes[0].set_ylabel("R2 Score")
axes[0].set_ylim(0, 1.1)
axes[0].tick_params(axis="x", rotation=10)

if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=features).sort_values()
    axes[1].barh(importances.index, importances.values, color="#66BB6A")
    axes[1].set_title(f"Feature Importance — {best_name}", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Importance Score")

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Best Model: {best_name}")
print(results_df.to_string(index=False))

## 15. Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, best_preds, alpha=0.3, s=10, color="#1565C0")
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
axes[0].plot(lims, lims, "r--", linewidth=1.2, label="Perfect prediction")
axes[0].set_xlabel("Actual Internet Penetration (%)")
axes[0].set_ylabel("Predicted Internet Penetration (%)")
axes[0].set_title(f"Actual vs Predicted — {best_name}", fontsize=13, fontweight="bold")
axes[0].legend()

residuals = y_test.values - best_preds
axes[1].hist(residuals, bins=40, color="#42A5F5", edgecolor="white", linewidth=0.3)
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.2)
axes[1].set_title("Residual Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Residual (%)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

## 16. Key Findings and Conclusions

### Global Trends
- Global internet penetration grew from roughly 7% in 2000 to over 65% in 2023
- The digital divide between regions remains significant — Europe and North America lead, Africa lags behind
- Middle East and Asia showed the fastest growth rates over the 24-year period

### Digital Divide
- UAE, South Korea, and Norway consistently rank at the top with 90%+ penetration
- Sub-Saharan African countries remain below 30% despite recent growth
- Income level is the strongest predictor of internet adoption

### Social Media
- High internet penetration strongly correlates with high social media usage
- UAE (100%) and Saudi Arabia (96%) lead social media penetration globally
- Countries with lower internet access also show lower social media adoption

### Machine Learning
- Previous year penetration (lag1) is the strongest predictor of current penetration
- Random Forest and Gradient Boosting achieve high R2 scores due to strong temporal patterns
- The model can be used to forecast future internet adoption for policy planning

---

Dataset by Hassan Ali | hassanali789 on Kaggle
Sources: World Bank · DataReportal 2023
